# Nemotron Jupyter and Colab Runner

This notebook runs the same training and evaluation pipeline from `app.py` in an interactive workflow.

Supported modes:
- TinyStories (default)
- JSONL chat data (including assistant-only loss masking)

In [ ]:
# Install dependencies.
# Local Jupyter: usually run this once in your virtualenv, then skip.
# Colab: run this cell every new session.

%pip install -q --upgrade pip
%pip install -q jax flax optax datasets transformers

In [ ]:
# Colab setup (optional): clone repo and move into it.
# Skip this cell if you already opened the notebook from the local repo.

import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    if not Path("nugie-jax-nemotron").exists():
        !git clone https://github.com/<your-username>/nugie-jax-nemotron.git
    %cd nugie-jax-nemotron
else:
    print("Running outside Colab. Current working directory:", Path.cwd())

In [ ]:
# Imports from project modules
import jax
import optax
from flax import nnx

from app import (
    load_hf_tokenizer,
    load_tinystories_texts,
    split_train_val_texts,
    load_jsonl_texts,
    prepare_datasets,
    create_lr_schedule,
    train_model,
    evaluate_model,
    generate_reply,
)
from nemotron import NemotronConfig, NemotronNanoBlock

In [ ]:
# Experiment configuration
cfg = {
    "preset": "tiny",
    "tokenizer_name": "google/byt5-small",
    "seed": 0,
    "steps": 20,
    "batch_size": 8,
    "seq_len": 64,
    "eval_batches": 5,
    "temperature": 0.0,
    "max_new_tokens": 80,

    # Dataset mode: "tinystories" or "jsonl"
    "dataset_format": "tinystories",

    # TinyStories settings
    "tinystories_max_stories": 2000,
    "tinystories_train_ratio": 0.9,
    "tinystories_split": "train",

    # JSONL settings
    "train_jsonl": "data/oasst2/train.jsonl",
    "val_jsonl": "data/oasst2/val.jsonl",
    "jsonl_text_key": "serialized_text",
    "jsonl_max_train_records": 50000,
    "jsonl_max_val_records": 5000,

    # Assistant-only masking (recommended for chat JSONL)
    "assistant_only_loss": False,
    "user_role_tag": "<|user|>",
    "assistant_role_tag": "<|assistant|>",
    "debug_mask_ratio": False,
    "debug_mask_every": 10,
}

print(cfg)

In [ ]:
# Load data
if cfg["dataset_format"] == "tinystories":
    all_stories = load_tinystories_texts(
        max_stories=cfg["tinystories_max_stories"],
        split=cfg["tinystories_split"],
        cache_dir=None,
    )
    train_texts, val_texts = split_train_val_texts(
        stories=all_stories,
        train_ratio=cfg["tinystories_train_ratio"],
    )
    print(
        f"Loaded TinyStories: total={len(all_stories)} train={len(train_texts)} val={len(val_texts)}"
    )
else:
    train_texts = load_jsonl_texts(
        jsonl_path=cfg["train_jsonl"],
        max_records=cfg["jsonl_max_train_records"],
        text_key=cfg["jsonl_text_key"],
    )
    val_texts = load_jsonl_texts(
        jsonl_path=cfg["val_jsonl"],
        max_records=cfg["jsonl_max_val_records"],
        text_key=cfg["jsonl_text_key"],
    )
    print(
        f"Loaded JSONL: train={len(train_texts)} val={len(val_texts)} key={cfg['jsonl_text_key']}"
    )

In [ ]:
# Build tokenizer, model, optimizer, and token streams
tokenizer = load_hf_tokenizer(cfg["tokenizer_name"], cache_dir=None)

model_cfg = NemotronConfig.from_preset(cfg["preset"])
model_cfg.vocab_size = len(tokenizer)

if cfg["seq_len"] % model_cfg.mamba_chunk_size != 0:
    raise ValueError(
        f"seq_len must be divisible by mamba_chunk_size ({model_cfg.mamba_chunk_size})"
    )

rngs = nnx.Rngs(cfg["seed"])
model = NemotronNanoBlock(rngs=rngs, config=model_cfg)

max_steps = max(cfg["steps"], 1)
warmup_steps = max(1, max_steps // 10)
lr_schedule = create_lr_schedule(max_steps=max_steps, warmup_steps=warmup_steps, peak_lr=3e-4)
tx = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.1),
)
optimizer = nnx.Optimizer(model, tx, wrt=nnx.Param)

train_tokens, val_tokens, train_loss_mask, val_loss_mask = prepare_datasets(
    tokenizer=tokenizer,
    train_texts=train_texts,
    val_texts=val_texts,
    seq_len=cfg["seq_len"],
    assistant_only_loss=cfg["assistant_only_loss"],
    user_role_tag=cfg["user_role_tag"],
    assistant_role_tag=cfg["assistant_role_tag"],
)

rng_key = jax.random.PRNGKey(cfg["seed"] + 1)
print("Prepared token streams.")

In [ ]:
# Train and evaluate
rng_key = train_model(
    model=model,
    optimizer=optimizer,
    train_tokens=train_tokens,
    train_loss_mask=train_loss_mask,
    steps=cfg["steps"],
    batch_size=cfg["batch_size"],
    seq_len=cfg["seq_len"],
    rng_key=rng_key,
    debug_mask_ratio=cfg["debug_mask_ratio"],
    debug_mask_every=cfg["debug_mask_every"],
)

mean_ce, perplexity, rng_key = evaluate_model(
    model=model,
    val_tokens=val_tokens,
    val_loss_mask=val_loss_mask,
    batch_size=cfg["batch_size"],
    seq_len=cfg["seq_len"],
    eval_batches=cfg["eval_batches"],
    rng_key=rng_key,
)

print({"mean_ce": mean_ce, "perplexity": perplexity})

In [ ]:
# Single-shot generation demo (not an interactive chat loop)
prompt = "Once upon a time"
reply, rng_key = generate_reply(
    model=model,
    tokenizer=tokenizer,
    prompt_text=prompt,
    seq_len=cfg["seq_len"],
    max_new_tokens=cfg["max_new_tokens"],
    temperature=cfg["temperature"],
    rng_key=rng_key,
)
print("Prompt:", prompt)
print("Reply:", reply.strip() or "...")